# Training the full hybrid retrieval-fusion pipeline on real Hebrew retail data

This notebook takes [`data.csv`](data.csv) from the `coicop_hebrew` example —
**~160k short Hebrew product names labeled into 186 in-house retail
categories** — and forgets about COICOP entirely. No taxonomy download, no
zero-shot step, no crosswalk. It is just: *here is labeled text, train a
calibrated, abstaining classifier on it, and see what results we get.*

It runs the [`text_classifier`](../../README.md) package's full pipeline —
five retrieval signals → ~28 fused features → XGBoost → isotonic calibration
→ a tuned abstention threshold — and is written for **Google Colab with a GPU
runtime**, so you can drop in a larger multilingual sentence-transformers
encoder than the lightweight one the packaged demo uses for speed, and
actually measure what the extra capacity buys you.

## What's in here

0. Setup — get the code + data into Colab
1. How the pipeline works (one paragraph + a picture in words)
2. Load and get oriented on `data.csv`
3. Data quality: missing labels, duplicates, label conflicts
4. Class imbalance — the reason this package exists
5. Clean the labeled set
6. Configure the run — fast dev pass vs. the full imbalanced dataset
7. Build the `LabelSpace` (the Hebrew label *is* the class — no richer gloss)
8. Choose and sanity-check the encoder
9. Pipeline configuration, explained knob by knob
10. Train the full pipeline
11. The trained model directory is self-describing
12. Held-out evaluation — the headline numbers
13. Calibration: is the confidence trustworthy?
14. The risk–coverage curve
15. Per-class breakdown — where does the imbalance bite?
16. Confusion analysis — what gets mixed up with what
17. Feature importance — which of the 28 signals drive the fusion model
18. The abstention threshold is a business dial
19. Optional: does the bigger encoder actually help? (small side-by-side)
20. Classify brand-new item strings
21. Save and download the trained model
22. Takeaways and open questions

Every section runs top to bottom; the "optional" ones are clearly marked and
safe to skip on a first pass.

## 0. Setup — get the code + data into Colab

**Use a GPU runtime**: `Runtime → Change runtime type → T4 GPU` (or better).
The encoder is called many times over the data (once per cross-validation
fold plus once for the final deployed index — see §1), so a bigger model
without a GPU can be very slow.

This cell tries, in order:

1. **Clone the repository** (the `dev` branch, where this example currently
   lives) and `pip install` the package from source.
2. If cloning fails (private repo, no network egress, ...), **upload a zip**
   of your own checkout instead — the cell below picks that up automatically.

Nothing here downloads an embedding model yet; that only happens in §8 when
you actually pick one.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/AmirShib/Hybrid-retrieval-fusion-text-classifier.git"
REPO_BRANCH = "dev"   # examples/coicop_hebrew (data.csv, items.csv) lives here
REPO_DIR = Path("Hybrid-retrieval-fusion-text-classifier")

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("running in Colab:", IN_COLAB)

if IN_COLAB and not (REPO_DIR / "text_classifier").is_dir():
    print(f"cloning {REPO_URL} (branch={REPO_BRANCH}) ...")
    result = subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, "--depth", "1", REPO_URL, str(REPO_DIR)],
        capture_output=True, text=True,
    )
    print(result.stdout[-2000:])
    if result.returncode != 0:
        print(result.stderr[-2000:])
        print(
            "\ngit clone failed -- the repo may be private, or this runtime has no "
            "network egress to it. Run the fallback upload cell below instead."
        )


In [ ]:
# Fallback: only needed if the clone above failed. Zip your own checkout
# (`cd Hybrid-retrieval-fusion-text-classifier && git archive -o repo.zip HEAD`,
# or GitHub's "Download ZIP") and upload it here; the folder name inside the
# zip doesn't matter, `find_repo_root` below searches for it.
if IN_COLAB and not (REPO_DIR / "text_classifier").is_dir():
    import zipfile
    from google.colab import files

    print("Upload a .zip of the repository checkout:")
    uploaded = files.upload()
    for name in uploaded:
        if name.lower().endswith(".zip"):
            with zipfile.ZipFile(name) as zf:
                zf.extractall(REPO_DIR)
            print(f"extracted {name} -> {REPO_DIR}/")
else:
    print("skipped (repo already present, or not running in Colab).")


In [ ]:
def find_repo_root(start: "Path | None" = None) -> Path:
    # Walk upward from `start` (default: cwd) looking for the package root, so
    # this works whether the notebook runs from a checkout you already have
    # open locally, or from a fresh Colab clone/upload one level below cwd.
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "text_classifier").is_dir():
            return candidate
    # Colab fallback: a fresh clone/zip-extract can land one or more levels
    # below cwd (e.g. inside a nested folder from a GitHub "Download ZIP"),
    # not necessarily as an immediate child, so search recursively.
    for pyproject in sorted(here.rglob("pyproject.toml")):
        if (pyproject.parent / "text_classifier").is_dir():
            return pyproject.parent
    raise RuntimeError(
        f"Could not locate the repository root from {here}. Did the clone/upload "
        "cell above finish successfully?"
    )


REPO_ROOT = find_repo_root()
EXAMPLE_DIR = REPO_ROOT / "examples" / "coicop_hebrew"
BUILD = EXAMPLE_DIR / "build"      # everything this notebook writes; git-ignored
BUILD.mkdir(parents=True, exist_ok=True)

print("repository root:", REPO_ROOT)
print("example dir    :", EXAMPLE_DIR)
assert (EXAMPLE_DIR / "data.csv").is_file(), "data.csv not found -- see the setup cells above"


In [ ]:
print("installing the package (brings in xgboost, scikit-learn, sentence-transformers/torch) ...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", str(REPO_ROOT)],
    capture_output=True, text=True,
)
print(result.stdout[-3000:])
if result.returncode != 0:
    print(result.stderr[-3000:])
    raise SystemExit("pip install failed -- see the error above.")
print("done.")


In [ ]:
import logging
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import text_classifier
from text_classifier import (
    PipelineConfig, LabelSpace, ClassDefinition, LabeledItem,
    TrainingPipeline, InferencePipeline,
)
from text_classifier.domain import FEATURE_NAMES
from text_classifier.infrastructure.encoder import SentenceTransformerEncoder
from text_classifier.application.evaluation import evaluate_decisions

logging.getLogger("text_classifier").setLevel(logging.INFO)
plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

SEED = 0
np.random.seed(SEED)

try:
    import torch
    HAS_CUDA = torch.cuda.is_available()
except ImportError:
    HAS_CUDA = False

DEVICE = "cuda" if HAS_CUDA else "cpu"
print("text_classifier version:", text_classifier.__version__)
print("Python:", sys.version.split()[0])
print("GPU available:", HAS_CUDA, "-> encoder device =", DEVICE)
if not HAS_CUDA:
    print("No GPU detected. Runtime -> Change runtime type -> GPU, then re-run this "
          "cell, or expect the encoder cells in <A7> onward to be much slower.")


## 1. How the pipeline works (one paragraph)

For every item, five retrieval signals are scored against the candidate
classes: dense (embedding) similarity to each class **description**, dense
similarity to each class **prototype** (the mean embedding of that class's
training examples), dense **k-NN** votes from the embedded training examples,
and the same description/k-NN pair computed **lexically** with BM25 instead of
embeddings. Each signal nominates its own top-N classes per item; the union is
the **candidate set**, so *candidate recall* (does the true class ever make
that set?) is the ceiling on system accuracy. The signals are turned into
~28 numeric features per (item, candidate) pair — similarities, ranks,
agreement counts, missingness flags (`NaN` means "this signal didn't retrieve
this class," never imputed away) — which a pointwise **XGBoost** model fuses
into a single relevance score per candidate. That score is **isotonic
calibrated** into `P(this candidate is correct)`, and a threshold (tuned to a
target accuracy on a held-out calibration fold) decides whether to emit the
top prediction or **abstain**.

Everything downstream in this notebook is built out-of-fold: an item's
features always come from an index built from *other* rows, so the coverage
and accuracy numbers you'll see are genuine held-out performance, not
training-set optimism.

Because we're forgetting COICOP, there is no external class taxonomy here —
the **186 Hebrew retail category names in `data.csv` are the classes**, used
both as the class *key* and (in place of a richer description) the class
*description*. That's a weaker description-similarity signal than the COICOP
demo's curated glosses, but it barely matters: with real labeled examples, the
prototype and k-NN signals — which only need data, not descriptions — carry
most of the weight (see §17).

## 2. Load and get oriented on `data.csv`

Columns: `lbcItemName` (the raw Hebrew product name, as scanned/exported from
a retail feed) and `label` (a Hebrew retail category name). We'll rename
`lbcItemName` -> `name` to match the rest of the package's vocabulary.

In [ ]:
raw = pd.read_csv(EXAMPLE_DIR / "data.csv").rename(columns={"lbcItemName": "name"})
raw = raw[["name", "label"]]
print(f"{len(raw):,} rows")
display(raw.head(10))
display(raw.sample(10, random_state=SEED))


## 3. Data quality — what's actually in here

Real exported retail data is never perfectly clean. Before training on it,
it's worth looking at three specific things: how much of it is even labeled,
how much is duplicated, and whether the same item name ever gets *two
different* labels (which would mean the "ground truth" itself is noisy —
a ceiling on achievable accuracy no model can beat).

In [ ]:
n_total = len(raw)
n_unlabeled = int(raw["label"].isna().sum())
n_empty_name = int((raw["name"].astype(str).str.strip() == "").sum())

name_stripped = raw["name"].astype(str).str.strip()
labeled = raw.assign(name=name_stripped).dropna(subset=["label"])
dup_rows = int(labeled.duplicated(subset=["name"]).sum())

conflict_mask = labeled.groupby("name")["label"].transform("nunique") > 1
n_conflict_rows = int(conflict_mask.sum())
n_conflict_names = int(labeled.loc[conflict_mask, "name"].nunique())

print(f"total rows                          : {n_total:,}")
print(f"missing label (unlabeled)           : {n_unlabeled:,}  ({100*n_unlabeled/n_total:.1f}%)")
print(f"blank/whitespace-only item name     : {n_empty_name:,}")
print(f"exact duplicate (name, label) rows*  : {dup_rows:,}  (* among labeled rows, by name)")
print(f"rows whose name has >1 distinct label: {n_conflict_rows:,}  across {n_conflict_names:,} distinct names")
print()
print("Examples of a name mapped to more than one label (labeling noise, not a bug):")
display(
    labeled[conflict_mask].sort_values("name").groupby("name").head(2).head(10)
)


**Reading this**: the unlabeled slice is simply out of scope for supervised
training (it's not "missing at random" — likely just not yet reviewed — so we
drop it rather than guess). The duplicate rows are largely repeat scans of the
same product and are harmless once we de-duplicate. The name/label conflicts
are genuine label noise (a couple hundred distinct names out of well over
100k): the same short string was filed under two different categories by
whoever labeled it, which is unavoidable at this taxonomy's granularity. We
drop those names entirely in §5 rather than arbitrarily picking one label,
and it sets an implicit ceiling: no model can be more consistent than the
labels it's trained to reproduce.

## 4. Class imbalance — the reason this package exists

`CLAUDE.md` describes this package as "built for imbalanced data." Let's look
at exactly how imbalanced `data.csv` is before deciding how to handle it.

In [ ]:
counts = labeled.dropna(subset=["label"])["label"].value_counts()
print(f"{len(counts)} distinct labels")
print(counts.describe().round(1))
print()
print("10 largest classes:")
display(counts.head(10).rename("count").to_frame())
print("10 smallest classes:")
display(counts.tail(10).rename("count").to_frame())


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(range(len(counts)), counts.to_numpy(), color="#457b9d", width=1.0)
ax.set_yscale("log")
ax.set_xlabel(f"class rank (1..{len(counts)}, sorted by frequency)")
ax.set_ylabel("item count (log scale)")
ax.set_title("Class frequency is heavy-tailed: a ~1,000x gap from largest to smallest")
plt.tight_layout()
plt.show()

print(f"largest class / smallest class ratio: {counts.max() / counts.min():.0f}x")


This is exactly the shape the package's `AbstentionPolicy` and per-class
threshold machinery exist for: a single global accuracy number can look fine
while the system is silently unreliable (or just abstaining constantly) on
the long tail. Keep an eye on the **per-class breakdown in §15** later, not
just the headline in §12 — that's where imbalance actually shows up.

`StratifiedKFold` (used internally for the leakage-free out-of-fold training
loop, `n_folds` folds by default) needs at least `n_folds` examples in every
class that's actually used for training. We'll filter out the handful of
classes below that floor in the next section rather than let `TrainingPipeline`
raise on them.

## 5. Clean the labeled set

Putting §3 and §4 together: drop unlabeled/blank rows, drop exact duplicate
`name` rows, drop names with conflicting labels, and drop classes with fewer
than `MIN_CLASS_COUNT` examples (comfortably above the cross-validation floor
from §4, not just at it — a class needs enough examples in the *train* split
too, not only enough to be technically splittable).

In [ ]:
MIN_CLASS_COUNT = 20  # comfortably above n_folds (5, see PipelineConfig in §9)

clean = raw.assign(name=raw["name"].astype(str).str.strip()).dropna(subset=["label"])
clean = clean[clean["name"].str.len() > 0]
clean = clean[~clean.groupby("name")["label"].transform("nunique").gt(1)]
clean = clean.drop_duplicates(subset=["name"]).reset_index(drop=True)

counts_clean = clean["label"].value_counts()
keep_labels = counts_clean[counts_clean >= MIN_CLASS_COUNT].index
dropped_classes = counts_clean[counts_clean < MIN_CLASS_COUNT]
clean = clean[clean["label"].isin(keep_labels)].reset_index(drop=True)

print(f"clean rows        : {len(clean):,}  (from {len(raw):,} raw rows)")
print(f"classes kept      : {clean['label'].nunique()}")
print(f"classes dropped   : {len(dropped_classes)}  "
      f"(fewer than {MIN_CLASS_COUNT} examples; {int(dropped_classes.sum())} rows lost)")


In [ ]:
lengths_chars = clean["name"].str.len()
lengths_words = clean["name"].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(lengths_chars, bins=40, color="#2a9d8f")
axes[0].set_title("item name length (characters)")
axes[1].hist(lengths_words, bins=range(1, 12), color="#e76f51")
axes[1].set_title("item name length (words)")
plt.tight_layout()
plt.show()

print(lengths_chars.describe().round(1))


Item names are short — a handful of words each, as expected from a retail
feed or receipt line. Short text is exactly where the lexical (BM25) signals
help less (little vocabulary overlap to exploit) and the dense/prototype
signals carry more weight; keep that in mind when reading §17's feature
importances.

## 6. Configure the run — fast dev pass vs. the full imbalanced dataset

`FAST_DEV_RUN = True` trains on a capped, more balanced subsample so you can
sanity-check the whole notebook quickly end to end. Flip it to `False` for the
real result: **all** cleaned data, real imbalance included, which is the
actual point of using this package rather than a plain classifier.

Compute cost scales with the number of cross-validation folds (default 5, see
§9): the out-of-fold loop calls the encoder's `.encode()` on roughly
`n_folds x len(train)` texts total (each fold both builds an index from its
training rows and embeds its validation rows), plus one more full pass to
build the final deployed index. The encoder's *weights* are not fine-tuned
per fold (that only happens if you explicitly set
`cfg.training.use_per_fold_encoder = True`) — it's the same frozen model
throughout — so this is a matter of repeated forward passes, not repeated
training. A bigger encoder or the full dataset multiplies that cost linearly;
a GPU (see §0) is what keeps it tractable.

In [ ]:
FAST_DEV_RUN = True   # flip to False once the dev pass looks sane

if FAST_DEV_RUN:
    MAX_CLASSES = 30        # cap on distinct classes used
    PER_CLASS_CAP = 80      # cap on examples per class (keeps the dev pass quick)
else:
    MAX_CLASSES = None       # None = use every class that survived §5
    PER_CLASS_CAP = None     # None = keep every example -- the real imbalance

pool = clean
if MAX_CLASSES is not None:
    top_labels = pool["label"].value_counts().head(MAX_CLASSES).index
    pool = pool[pool["label"].isin(top_labels)]
if PER_CLASS_CAP is not None:
    # A plain groupby-iterate + concat (rather than groupby(...).apply(...)) sidesteps
    # a pandas-version difference in whether the grouping column survives .apply().
    pool = pd.concat(
        [g.sample(min(len(g), PER_CLASS_CAP), random_state=SEED) for _, g in pool.groupby("label")],
        ignore_index=True,
    )
pool = pool.reset_index(drop=True)

from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    pool, test_size=0.2, stratify=pool["label"], random_state=SEED
)
print(f"FAST_DEV_RUN={FAST_DEV_RUN}  ->  {pool['label'].nunique()} classes, "
      f"{len(train_df):,} train / {len(test_df):,} held-out test rows")
display(pool["label"].value_counts().head(8).rename("n_items").to_frame())


## 7. Build the `LabelSpace` — the Hebrew label *is* the class

No COICOP codes, no crosswalk. Each distinct Hebrew label becomes a
`ClassDefinition(key=label, description=label)` — the key and the description
are the same string, since we have no richer gloss to fold in (contrast the
COICOP demo's `classes.csv`, which packs in official titles plus "includes"
notes). That weakens the description-similarity signal specifically, but not
the pipeline overall: with labeled examples, the class **prototype** (mean
embedding of that class's training items) and the dense/BM25 **k-NN** signals
are built from real item text and do the heavy lifting instead.

In [ ]:
label_space = LabelSpace([ClassDefinition(k, k) for k in sorted(pool["label"].unique())])
train_items = [LabeledItem(text, label) for text, label in zip(train_df["name"], train_df["label"])]

print(f"{label_space.size} classes, {len(train_items)} labeled training items")
print("first few class keys:", label_space.keys[:5])


## 8. Choose and sanity-check the encoder

The packaged COICOP demo uses `paraphrase-multilingual-MiniLM-L12-v2` (~470 MB,
118M params) purely for notebook speed. This is the cell to swap in something
bigger:

| model | size / params | notes |
|---|---|---|
| `paraphrase-multilingual-MiniLM-L12-v2` | ~470 MB / 118M | fast, the packaged demo's default |
| `paraphrase-multilingual-mpnet-base-v2` | ~1.1 GB / 278M | stronger, still general-purpose |
| `sentence-transformers/LaBSE` | ~1.8 GB / 471M | translation-pair trained, very robust cross-lingual (used for zero-shot in the COICOP demo) |
| `intfloat/multilingual-e5-large` | ~2.2 GB / 560M | strong on retrieval benchmarks, but expects `"query: "` / `"passage: "` prefixes on its inputs for best results -- this package's `SentenceTransformerEncoder` does *not* add them automatically, so either prefix `name`/`label` text yourself before it reaches the pipeline, or expect it to underperform its usual benchmark numbers here |

Set `MODEL_NAME` below. We load the encoder **once** here (not inside the
training pipeline) and hand this exact instance to `TrainingPipeline` via its
`shared_encoder=` argument in §10 -- that avoids loading the same multi-hundred-
MB model twice.

In [ ]:
MODEL_NAME = "sentence-transformers/LaBSE"   # <- change this to try a different encoder
BATCH_SIZE = 64

print(f"loading {MODEL_NAME} on {DEVICE} (first run downloads the weights) ...")
t0 = time.perf_counter()
encoder = SentenceTransformerEncoder.load(MODEL_NAME, batch_size=BATCH_SIZE, device=DEVICE)
print(f"loaded in {time.perf_counter() - t0:.1f}s")

# Sanity check: two items of the *same* class should sit closer in embedding
# space than two items of *different* classes, before we spend any training
# compute on this encoder.
by_label = train_df.groupby("label")["name"].apply(list)
big_classes = by_label[by_label.str.len() >= 2].index
a_label, b_label = np.random.RandomState(SEED).choice(big_classes, size=2, replace=False)
a0, a1 = by_label[a_label][:2]     # two items of the same class
b0 = by_label[b_label][0]          # one item of a different class

probe = encoder.encode([a0, a1, b0])
sim_same = float(probe[0] @ probe[1])
sim_diff = float(probe[0] @ probe[2])
print(f"\nsame-class pair  ({a_label!r}): {a0!r} / {a1!r} -> cos = {sim_same:.3f}")
print(f"diff-class pair  ({a_label!r} vs {b_label!r}): {a0!r} / {b0!r} -> cos = {sim_diff:.3f}")
print("(same-class should usually score higher; if it doesn't, this encoder may "
      "be a weak fit for short Hebrew retail text -- try a different MODEL_NAME.)")


## 9. Pipeline configuration, explained knob by knob

`PipelineConfig` is a plain, JSON-serializable set of dataclasses (it's
written into the model directory alongside the trained artifacts, so a
deployed model documents exactly how it was trained). The knobs that matter
most for this dataset:

- **`encoder.kind` / `model_name_or_path`** -- recorded for provenance; the
  actual instance used for training is the `shared_encoder` passed to
  `TrainingPipeline` below, so this doesn't trigger a second load.
- **`retrieval.k_neighbors`** -- how many nearest labeled examples each dense/
  BM25 k-NN signal looks at per item.
- **`fusion.kind`** -- `"xgboost"` (default, used here), `"lightgbm"`, or
  `"xgbranker"` (learning-to-rank) are all drop-in swappable.
- **`calibration.kind`** -- `"isotonic"` (default, non-parametric, needs more
  calibration data), `"platt"` and `"beta"` (parametric, more robust with a
  smaller calibration fold) are drop-in swappable too.
- **`training.n_folds`** -- cross-validation folds for the leakage-free
  out-of-fold loop; the *last* fold becomes the untouched test set, the
  *second-to-last* becomes the calibration set, the rest train the fusion
  model (`TrainingConfig.fold_roles`).
- **`training.target_precision`** -- the business lever: "only accept a
  prediction when the historical accuracy at that confidence level is >= this."
  Lower it for more coverage at the cost of accuracy on accepted items, or
  raise it for the reverse (see the sweep in §18).
- **`training.per_class_min_support`** -- a class only gets its *own* tuned
  threshold if its calibration-fold support clears this bar; otherwise it
  falls back to the global threshold.
- **`candidate_top_n`** -- how many classes each of the five signals may
  nominate into the candidate set; the union across signals is what
  `candidate_recall` (§12) measures the ceiling of.

In [ ]:
cfg = PipelineConfig()
cfg.encoder.kind = "sentence-transformers"
cfg.encoder.model_name_or_path = MODEL_NAME
cfg.encoder.encode_batch_size = BATCH_SIZE
cfg.encoder.device = DEVICE

cfg.fusion.kind = "xgboost"
cfg.calibration.kind = "isotonic"

cfg.training.n_folds = 5
cfg.training.target_precision = 0.90   # accept only where accuracy is expected >= 90%
cfg.training.per_class_min_support = 50
cfg.training.use_per_fold_encoder = False   # frozen pretrained encoder, no need to refit per fold

cfg.candidate_top_n = 10

print(cfg)


## 10. Train the full pipeline

`TrainingPipeline.run` does the whole job: out-of-fold feature generation ->
fusion model fit -> calibration + threshold tuning -> held-out evaluation ->
final encoder + indices fit on all data, saved to `MODEL_DIR`. Passing our
already-loaded `encoder` as `shared_encoder` means the multi-hundred-MB model
is loaded once total, not once per pipeline instantiation.

This is the expensive cell in the notebook -- see the compute-cost note in
§6 if it's taking a long time; `FAST_DEV_RUN`, `MODEL_NAME`, and
`cfg.fusion.xgb_params["n_estimators"]` are the main levers.

In [ ]:
MODEL_DIR = BUILD / "hebrew_full_model"

print(f"training on {len(train_items):,} items across {label_space.size} classes ...")
t0 = time.perf_counter()
artifacts, report = TrainingPipeline(cfg, shared_encoder=encoder).run(
    train_items, label_space, output_dir=str(MODEL_DIR)
)
elapsed = time.perf_counter() - t0
print(f"done in {elapsed/60:.1f} min.")

pd.Series({
    "candidate recall (accuracy ceiling)": report.candidate_recall,
    "coverage (accepted fraction)":        report.coverage,
    "accuracy on accepted":                report.accuracy_on_accepted,
    "accuracy if never abstaining":        report.accuracy_if_no_abstain,
}).round(3).to_frame("out-of-fold test fold")


Read this by row: **candidate recall** is the ceiling -- if the true
class doesn't even make the candidate shortlist, no amount of fusion/
calibration can recover it (a low number here means raising
`candidate_top_n` or `k_neighbors`, not tuning the classifier further).
**Accuracy on accepted** is what a downstream consumer actually experiences,
and it should sit meaningfully above **accuracy if never abstaining** -- that
gap is the abstention policy buying precision by declining the hardest cases
instead of guessing on them.

## 11. The trained model directory is self-describing

Every trained directory carries its own evidence, so it can be audited later
without rerunning anything: `model_card.md` is the human-readable summary,
`evaluation.json` holds the full per-class breakdown, calibration
diagnostics, and risk-coverage curve, and `meta.json` records the exact
`PipelineConfig` used. This is also exactly what ships to an air-gapped host
(stdlib pickle + numpy + JSON + native XGBoost/SentenceTransformer formats
only -- see `CLAUDE.md`).

In [ ]:
print("files in the model directory:")
for path in sorted(MODEL_DIR.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(MODEL_DIR)}  ({path.stat().st_size:,} bytes)")

print("\n--- model_card.md ---\n")
print((MODEL_DIR / "model_card.md").read_text())


## 12. Held-out evaluation — the headline numbers

`TrainingPipeline`'s report above already comes from a held-out fold, but
let's also close the loop the way a deployment would: reload the saved
directory through `InferencePipeline` (nothing in memory carried over) and
score the untouched `test_df` split we set aside all the way back in §6 --
this data was never used to build any index, train the fusion model, or
calibrate it.

In [ ]:
pipe = InferencePipeline.from_directory(str(MODEL_DIR))
keys = pipe.label_space.keys
key_to_idx = {k: i for i, k in enumerate(keys)}

test_preds = pipe.predict(list(test_df["name"]))

confidence = np.array([p.confidence for p in test_preds])
accepted = np.array([not p.abstained for p in test_preds])
pred_idx = np.array([key_to_idx.get(p.top_key, -1) for p in test_preds])
true_idx = np.array([key_to_idx[l] for l in test_df["label"]])
correct = (pred_idx == true_idx)

evaluation = evaluate_decisions(
    confidence=confidence, correct=correct, accepted=accepted,
    pred_idx=pred_idx, true_idx=true_idx, keys=keys,
)
o, calib = evaluation["overall"], evaluation["calibration"]
pd.Series({
    "items evaluated":        o["n_items"],
    "coverage":               o["coverage"],
    "accuracy on accepted":   o["accuracy_on_accepted"],
    "accuracy if no abstain": o["accuracy_if_no_abstain"],
    "Brier score":            calib["brier_score"],
    "expected calib. error":  calib["expected_calibration_error"],
}).round(4).to_frame("held-out test (§6 split)")


In [ ]:
# One row per test item -- reused by the per-class and confusion sections below.
results_df = pd.DataFrame({
    "name": test_df["name"].to_numpy(),
    "true_label": test_df["label"].to_numpy(),
    "pred_label": np.where(pred_idx >= 0, np.asarray(keys)[np.clip(pred_idx, 0, None)], "<no candidate>"),
    "confidence": confidence,
    "accepted": accepted,
    "correct": correct,
})
display(results_df.sample(10, random_state=SEED))


## 13. Calibration: is the confidence trustworthy?

The abstention threshold in §9/§18 is only meaningful if the confidence it
acts on is *calibrated* -- items the model is "90% sure" about should really
be right about 90% of the time. The reliability diagram bins items by
confidence and plots mean confidence against observed accuracy per bin;
perfect calibration sits on the diagonal.

In [ ]:
rel = pd.DataFrame(calib["reliability_table"])

fig, ax = plt.subplots()
ax.plot([0, 1], [0, 1], "--", color="grey", label="perfect calibration")
ax.plot(rel["mean_confidence"], rel["accuracy"], "o-", color="#1f77b4", label="model")
sizes = 300 * rel["count"] / rel["count"].max()
ax.scatter(rel["mean_confidence"], rel["accuracy"], s=sizes, color="#1f77b4", alpha=0.3)
ax.set_xlabel("mean calibrated confidence (per bin)")
ax.set_ylabel("observed accuracy (per bin)")
ax.set_title(f"Reliability diagram  (ECE = {calib['expected_calibration_error']:.4f})")
ax.legend()
plt.tight_layout()
plt.show()

display(rel.round(3))


## 14. The risk–coverage curve

The central trade-off for a human-in-the-loop system: as you accept fewer
items (lower coverage), the ones you *do* accept should get more accurate
(lower risk). This curve is the model-independent ceiling on what any single
global confidence threshold can achieve -- it's what a reviewer reads to pick
an operating point, and what `target_precision` (§9) picks a point on
automatically.

In [ ]:
rc = pd.DataFrame(evaluation["risk_coverage_curve"])

fig, ax = plt.subplots()
ax.plot(rc["coverage"], rc["accuracy"], "o-", color="#2ca02c")
ax.axhline(cfg.training.target_precision, color="#e76f51", ls="--", lw=1,
           label=f"target_precision = {cfg.training.target_precision}")
ax.set_xlabel("coverage (fraction of items accepted)")
ax.set_ylabel("accuracy on accepted items")
ax.set_title("Risk-coverage: accuracy rises as the model accepts fewer, surer items")
ax.set_ylim(min(0.5, rc["accuracy"].min() - 0.02), 1.02)
ax.legend()
plt.tight_layout()
plt.show()


## 15. Per-class breakdown — where does the imbalance bite?

A healthy global accuracy number can hide classes the system silently
abstains on entirely, or systematically confuses with a look-alike category.
This is the table `evaluate_decisions` already computes but the headline in
§12 doesn't show -- exactly the view that matters on data this imbalanced
(§4).

In [ ]:
per_class = pd.DataFrame(evaluation["per_class"])
print(f"{len(per_class)} classes appear as a true label or a prediction in the test set")
display(per_class.head(10))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(
    per_class["support"], per_class["recall_on_accepted"].fillna(0),
    s=30, c=per_class["coverage"].fillna(0), cmap="viridis", alpha=0.8,
)
ax.set_xscale("log")
ax.set_xlabel("class support in test set (log scale)")
ax.set_ylabel("recall on accepted (of items whose true class is this)")
ax.set_title("Per-class recall vs. support, colored by coverage")
plt.colorbar(sc, label="coverage")
plt.tight_layout()
plt.show()


In [ ]:
MIN_SUPPORT_TO_JUDGE = 5   # ignore classes too small in the test set to read much into
judged = per_class[per_class["support"] >= MIN_SUPPORT_TO_JUDGE]

print("Worst 15 classes by recall-on-accepted (support >= {}):".format(MIN_SUPPORT_TO_JUDGE))
display(judged.sort_values("recall_on_accepted").head(15))

print("\nBest 15 classes by recall-on-accepted (support >= {}):".format(MIN_SUPPORT_TO_JUDGE))
display(judged.sort_values("recall_on_accepted", ascending=False).head(15))


If the worst classes here are also the smallest (low `support`), that's
the imbalance showing up exactly where you'd expect -- the model simply has
fewer examples to build a prototype/k-NN neighborhood from. If a *large*
class scores poorly, that's more interesting: look at its confusions in §16
next, it's likely overlapping semantically with a sibling category rather
than being under-supported.

## 16. Confusion analysis — what gets mixed up with what

With 30-186 classes a full confusion matrix is unreadable. More useful: the
most common (true, predicted) mistake pairs among **accepted** (not
abstained) wrong predictions -- these are the confusions that actually reach
a downstream consumer, not just any wrong candidate.

In [ ]:
wrong = results_df[results_df["accepted"] & ~results_df["correct"]]
confused_pairs = (
    wrong.groupby(["true_label", "pred_label"])
    .size().rename("n").sort_values(ascending=False).head(20).reset_index()
)
print(f"{len(wrong)} accepted-but-wrong predictions in the test set "
      f"(out of {int(results_df['accepted'].sum())} accepted)")
display(confused_pairs)


Pairs that recur here are usually genuinely close categories (e.g. two
flavors of the same product type, or a category boundary that's fuzzy even
for a human labeler) -- worth checking against the §3 label-conflict list,
since a name that's ambiguous in the source data will look exactly like a
"confusion" here even though no model could resolve it. If a pair is *not*
an obvious near-duplicate, it's a genuine model weakness -- candidate for
richer class descriptions, more examples, or a coarser taxonomy at that
boundary.

## 17. Feature importance — which of the 28 signals drive the fusion model

The fusion model consumes the ~28 columns in `FEATURE_NAMES`
(`domain/services.py`) -- dense/BM25 similarities, ranks, agreement counts,
missingness flags. For the default `"xgboost"` backend, XGBoost's gain-based
importances tell us which of the five retrieval signals the fusion model
actually leans on for *this* dataset. (This reaches into the fusion
backend's internals, which is fine for a one-off diagnostic like this but
isn't part of the package's public API.)

In [ ]:
xgb_model = getattr(artifacts.fusion, "_model", None)
if xgb_model is not None and hasattr(xgb_model, "feature_importances_"):
    importances = pd.Series(xgb_model.feature_importances_, index=FEATURE_NAMES).sort_values()
    fig, ax = plt.subplots(figsize=(7, 8))
    ax.barh(importances.index, importances.to_numpy(), color="#457b9d")
    ax.set_xlabel("XGBoost gain-based importance")
    ax.set_title("Which of the 28 fusion features drive the decision?")
    plt.tight_layout()
    plt.show()
    print("\ntop 5:")
    print(importances.tail(5).iloc[::-1])
else:
    print(f"feature_importances_ isn't available for fusion.kind={cfg.fusion.kind!r}; "
          "this diagnostic only applies to the xgboost backend.")


## 18. The abstention threshold is a business dial

`target_precision` in §9 already picked *one* operating point automatically.
But the threshold is a real choice: a lower one accepts more traffic at lower
accuracy, a higher one is the reverse. Sweeping it over the held-out test
confidences makes the trade-off concrete (this uses the single *global*
threshold value for the sweep -- the deployed model may also carry per-class
overrides from §9's `per_class_min_support`, so its real behavior is slightly
finer-grained than this chart).

In [ ]:
thresholds = np.linspace(0.0, 1.0, 101)

def coverage_at(t):
    return float((confidence >= t).mean())

def accuracy_on_accepted_at(t):
    m = confidence >= t
    return float(correct[m].mean()) if m.any() else np.nan

fig, ax1 = plt.subplots()
cov = [coverage_at(t) for t in thresholds]
acc = [accuracy_on_accepted_at(t) for t in thresholds]
ax1.plot(thresholds, cov, color="#1f77b4", label="coverage")
ax1.set_xlabel("confidence threshold")
ax1.set_ylabel("coverage", color="#1f77b4")
ax2 = ax1.twinx()
ax2.plot(thresholds, acc, color="#e76f51", label="accuracy on accepted")
ax2.set_ylabel("accuracy on accepted", color="#e76f51")
ax1.axvline(artifacts.abstention.global_threshold, color="grey", ls="--", lw=1,
            label="trained global threshold")
fig.legend(loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.05))
plt.title("Coverage vs. accuracy as the confidence threshold moves")
plt.tight_layout()
plt.show()


In [ ]:
def summarize(t):
    m = confidence >= t
    return {
        "threshold":            round(float(t), 3),
        "coverage":              f"{100 * coverage_at(t):.1f}%",
        "accuracy on accepted":  f"{100 * accuracy_on_accepted_at(t):.1f}%" if m.any() else "n/a",
    }

pd.DataFrame([summarize(t) for t in [0.0, 0.3, 0.5, artifacts.abstention.global_threshold, 0.7, 0.9]])


## 19. Optional: does the bigger encoder actually help? (small side-by-side)

The point of this notebook was to plug in a larger embedding model and see
what it buys you. This section trains two small, matched pipelines -- the
packaged demo's lightweight `MiniLM` vs. your `MODEL_NAME` from §8 -- on the
**same** reduced subsample, so the only thing that differs is the encoder.
Kept deliberately small (`CMP_MAX_CLASSES` / `CMP_PER_CLASS`) so this is a
quick sanity check, not the main event -- the real comparison is the full run
in §10-18 above versus rerunning this whole notebook with `MODEL_NAME` set
back to `MiniLM`.

Safe to skip this section entirely if you're only interested in the one
encoder you already chose.

In [ ]:
RUN_ENCODER_COMPARISON = True

if RUN_ENCODER_COMPARISON:
    CMP_MAX_CLASSES, CMP_PER_CLASS = 20, 60
    cmp_top = clean["label"].value_counts().head(CMP_MAX_CLASSES).index
    cmp_pool = clean[clean["label"].isin(cmp_top)]
    cmp_pool = pd.concat(
        [g.sample(min(len(g), CMP_PER_CLASS), random_state=SEED) for _, g in cmp_pool.groupby("label")],
        ignore_index=True,
    )
    cmp_train, cmp_test = train_test_split(
        cmp_pool, test_size=0.2, stratify=cmp_pool["label"], random_state=SEED
    )
    cmp_label_space = LabelSpace([ClassDefinition(k, k) for k in sorted(cmp_pool["label"].unique())])
    cmp_items = [LabeledItem(t, l) for t, l in zip(cmp_train["name"], cmp_train["label"])]
    print(f"comparison pool: {cmp_label_space.size} classes, "
          f"{len(cmp_train)} train / {len(cmp_test)} test rows")
else:
    print("skipped.")


In [ ]:
def train_and_eval(model_name, out_dir):
    enc = SentenceTransformerEncoder.load(model_name, batch_size=BATCH_SIZE, device=DEVICE)
    c = PipelineConfig()
    c.encoder.kind, c.encoder.model_name_or_path = "sentence-transformers", model_name
    c.encoder.device = DEVICE
    c.training.target_precision = cfg.training.target_precision
    t0 = time.perf_counter()
    _, rep = TrainingPipeline(c, shared_encoder=enc).run(cmp_items, cmp_label_space, output_dir=str(out_dir))
    return {
        "model": model_name,
        "seconds": round(time.perf_counter() - t0, 1),
        "candidate_recall": round(rep.candidate_recall, 3),
        "coverage": round(rep.coverage, 3),
        "accuracy_on_accepted": round(rep.accuracy_on_accepted, 3),
        "accuracy_if_no_abstain": round(rep.accuracy_if_no_abstain, 3),
    }

if RUN_ENCODER_COMPARISON:
    comparison = pd.DataFrame([
        train_and_eval(
            "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
            BUILD / "cmp_minilm",
        ),
        train_and_eval(MODEL_NAME, BUILD / "cmp_chosen_model"),
    ])
    display(comparison)
    print("\nSame subsample, same target_precision -- any gap in `coverage` / "
          "`accuracy_on_accepted` / `candidate_recall` here is attributable to "
          "the encoder alone. `seconds` is single-run wall-clock on this runtime, "
          "not a robust benchmark.")


## 20. Classify brand-new item strings

[`items.csv`](items.csv) has 15 short Hebrew grocery names that were written
for the *zero-shot COICOP* half of the packaged demo, unlabeled here too --
a good concrete check that the model trained on §10's Hebrew taxonomy
generalizes to text it has never seen, not just the held-out test split.

In [ ]:
new_items = pd.read_csv(EXAMPLE_DIR / "items.csv")["name"].astype(str).tolist()
new_items += [
    "יין אדום קברנה סוביניון",   # a couple of hand-written extras
    "מגבונים לחים לתינוק",
]

new_preds = pipe.predict(new_items)
pd.DataFrame({
    "item": new_items,
    "prediction": [p.predicted_key if not p.abstained else "— ABSTAIN —" for p in new_preds],
    "confidence": [round(p.confidence, 3) for p in new_preds],
    "runner-up": [p.runner_up_key for p in new_preds],
})


## 21. Save and download the trained model

`MODEL_DIR` is a complete, portable model directory -- stdlib pickle + numpy
+ JSON + native XGBoost/SentenceTransformer formats only, so it also ships to
an air-gapped host per `CLAUDE.md`'s invariants. Zip it and pull it down
locally.

In [ ]:
import shutil

zip_path = shutil.make_archive(
    str(BUILD / MODEL_DIR.name), "zip", root_dir=MODEL_DIR.parent, base_dir=MODEL_DIR.name
)
print(f"zipped -> {zip_path}  ({Path(zip_path).stat().st_size / 1e6:.1f} MB)")

if IN_COLAB:
    from google.colab import files
    files.download(zip_path)
else:
    print("not running in Colab -- the zip is at the path above.")


## 22. Takeaways and open questions

**What this run showed** (fill in with your own numbers from §10-19 above):

- Candidate recall (§10) is the honest ceiling on accuracy -- if it's low,
  raising `k_neighbors` / `candidate_top_n` matters more than tuning the
  fusion model further.
- The gap between accuracy-on-accepted and accuracy-if-never-abstaining
  (§10, §12) is what calibrated abstention is actually buying you.
- The reliability diagram (§13) tells you whether the abstention threshold
  is trustworthy, not just whether the headline accuracy looks good.
- The per-class table (§15) and confusion pairs (§16) are where imbalance
  and near-duplicate categories actually surface -- the global numbers in
  §12 can hide both.
- The feature importances (§17) show whether this dataset needs the
  description-similarity signal at all, given the classes here have no
  richer gloss than their own name (contrast the COICOP demo's curated
  descriptions).

**Open questions worth trying next:**

- **Richer class descriptions.** Right now each class's "description" is
  just its own Hebrew name (§7). Concatenating a handful of representative
  item names per class into a synthetic description (`"typical items: ..."`)
  would give the description-similarity signals something more to work with
  -- cheap to try, since it only touches how `label_space` is built.
- **Encoder fine-tuning.** `text_classifier.infrastructure.encoder.train_encoder`
  fine-tunes a sentence-transformer on `(item, description)` pairs via
  `cfg.training.use_per_fold_encoder = True`. Worth it once you've settled on
  a base model, expensive to run inside every fold -- try it as a separate
  experiment on the final full-data encoder rather than the cross-validation
  loop.
- **Per-class vs. global threshold.** `per_class_min_support` (§9) controls
  how many classes get their own tuned threshold instead of falling back to
  the global one -- lowering it trades threshold precision for broader
  per-class coverage; the abstention section of `evaluation.json` records how
  many classes actually got one.
- **Alternative fusion/calibration backends.** `cfg.fusion.kind` also
  accepts `"lightgbm"` and `"xgbranker"` (learning-to-rank); `cfg.calibration.kind`
  also accepts `"platt"` and `"beta"` -- both are one-line swaps worth
  A/B-ing against the isotonic/XGBoost defaults used here.
- **The label-conflict ceiling (§3).** A few hundred item names carry more
  than one label in the raw data -- that's a hard ceiling on achievable
  accuracy for those specific names no matter how good the model gets.
  Resolving the conflicts upstream (majority vote, or a taxonomy review) would
  lift it.
- **Back to COICOP, if you ever need it.** Nothing here produces COICOP codes
  -- this Hebrew retail taxonomy is a separate, real-world label set. If
  COICOP codes are the actual target, either crosswalk each Hebrew label to a
  COICOP code once (`item -> Hebrew label -> COICOP`, many-to-one is fine)
  and retrain straight onto COICOP keys, or use this trained encoder's
  domain-tuned embeddings to sharpen the *zero-shot* COICOP step in the
  original `coicop_hebrew_classification.ipynb` instead.